In [11]:
#import the library
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from scipy.stats import loguniform, randint
from imblearn.pipeline import Pipeline as ImbPipeline


In [18]:
#Defining the pre-model hyperparameter search spaces
MODEL_SPECS = {
    "LogisticRegression": {
        "estimator": lambda random_state: LogisticRegression(
            max_iter=5000, random_state=random_state
        ),
        "param_distributions": {
            "model__C":            loguniform(1e-3, 1e2),
            "model__penalty":      ["l1", "l2"],
            "model__solver":       ["liblinear"],
            "model__class_weight": [None, "balanced"],
        },
        "n_iter": 50,
    },
    "RandomForest": {
        "estimator": lambda random_state: RandomForestClassifier(
            random_state=random_state, n_jobs=-1
        ),
        "param_distributions": {
            "model__n_estimators":      randint(100, 500),
            "model__max_depth":         [2, 3, 4, 5, 6, None],
            "model__min_samples_split": randint(2, 10),
            "model__min_samples_leaf":  randint(1, 6),
            "model__max_features":      ["sqrt", "log2", None],
            "model__class_weight":      [None, "balanced"],
        },
        "n_iter": 50,
    }
}

In [3]:
def build_pipeline(estimator, random_state, use_smote):
    steps = [("scaler", RobustScaler())]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=random_state)))
    steps.append(("model", estimator))
    return ImbPipeline(steps=steps)



In [7]:
# 3. Nested CV — corrected pipeline
def nested_cv_corrected(
    model_name, X, y,
    n_outer_splits=5, n_inner_splits=3, random_state=42, use_smote=True,
):
    spec = MODEL_SPECS[model_name]

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    X_values = X.values
    y_values = y.values if isinstance(y, pd.Series) else np.asarray(y)

    outer_skf = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                 random_state=random_state)

    results = {"roc_auc": [], "accuracy": [], "sensitivity": [], "specificity": [],
               "best_params": []}

    print(f"\n{'='*60}\n{model_name}: Nested CV ({n_outer_splits}x{n_inner_splits}, corrected pipeline)\n{'='*60}")
    total_start = time.time()

    for outer_fold, (train_idx, test_idx) in enumerate(
            outer_skf.split(X_values, y_values), 1):

        fold_start = time.time()

        # ---- Step 1: split outer train / outer test first, nothing else touches test yet
        X_train_outer, X_test_outer = X_values[train_idx], X_values[test_idx]
        y_train_outer, y_test_outer = y_values[train_idx], y_values[test_idx]

        # ---- Step 2: inner 3-fold CV over the OUTER-TRAINING data only.
        # RandomizedSearchCV clones and re-fits `pipe` on each inner-training
        # split — i.e. RobustScaler.fit_transform + SMOTE.fit_resample are
        # each recomputed from scratch on that inner-training fold only,
        # and only .transform() (no SMOTE) is applied to the inner-validation
        # fold. This is exactly the top branch of your diagram.
        inner_skf = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                                     random_state=random_state)

        pipe = build_pipeline(spec["estimator"](random_state), random_state, use_smote)

        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=spec["param_distributions"],
            n_iter=spec["n_iter"],
            cv=inner_skf,
            scoring="roc_auc",
            n_jobs=-1,
            random_state=random_state,
            verbose=0,
            refit=True,   # <-- after search, refits the WINNING pipeline
                          #     (scaler + SMOTE + model) on the COMPLETE
                          #     outer-training partition. This is exactly
                          #     the bottom branch of your diagram.
        )
        search.fit(X_train_outer, y_train_outer)

        best_pipeline = search.best_estimator_   # scaler+SMOTE+model, refit on full outer-train
        best_params   = search.best_params_

        # ---- Step 3: evaluate ONCE on the untouched outer test fold.
        # best_pipeline.predict()/.predict_proba() apply scaler.transform()
        # (fitted on outer-train) and skip SMOTE entirely (SMOTE never runs
        # outside .fit()) — so the outer test fold is genuinely untouched
        # by both scaling-fit and resampling.
        y_pred = best_pipeline.predict(X_test_outer)
        y_prob = best_pipeline.predict_proba(X_test_outer)[:, 1]

        cm = confusion_matrix(y_test_outer, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        results["roc_auc"].append(roc_auc_score(y_test_outer, y_prob))
        results["accuracy"].append(accuracy_score(y_test_outer, y_pred))
        results["sensitivity"].append(tp / (tp + fn) if (tp + fn) else np.nan)
        results["specificity"].append(tn / (tn + fp) if (tn + fp) else np.nan)
        results["best_params"].append(best_params)

        print(f"  Fold {outer_fold}: ROC-AUC={results['roc_auc'][-1]:.4f}  "
              f"Acc={results['accuracy'][-1]:.4f}  "
              f"Sens={results['sensitivity'][-1]:.4f}  "
              f"Spec={results['specificity'][-1]:.4f}  "
              f"({time.time()-fold_start:.1f}s)  params={best_params}")

    print(f"\n{model_name} — FINAL (mean ± std across {n_outer_splits} folds):")
    for metric in ["roc_auc", "accuracy", "sensitivity", "specificity"]:
        m, sd = np.mean(results[metric]), np.std(results[metric])
        print(f"  {metric:12s}: {m:.4f} ± {sd:.4f}")
    print(f"  Total time: {time.time()-total_start:.1f}s")

    return results

In [8]:
from google.colab import files
uploaded = files.upload()

Saving medical_data_clean.csv to medical_data_clean.csv


In [9]:
#Reading the data
df = pd.read_csv('medical_data_clean.csv')
display(df.head())

#Defining the X and Y variables
X = df.drop(columns=["Group","LC_stage","LC_type","lscm","Age","Sex","Smoking","Group_name"])
y = df["Group"]
display(X.head())
display(y.head())

,S1_T1,S2_T1,S3_T1,S4_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S4_T2,...,S5_T3,S6_T3,Age,Sex,Smoking,LC_stage,LC_type,Group,Group_name,lscm
0,3.180000e-07,3.730000e-07,0.000230,9.420000e-07,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000004,...,8.850000e-07,7.660000e-07,50,1,1,3,0,1,LC group,4
1,3.300000e-07,2.770000e-07,0.000268,1.120000e-06,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000002,...,1.160000e-06,9.310000e-07,68,0,0,4,1,1,LC group,3
2,5.450000e-07,4.810000e-07,0.000527,2.400000e-06,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000005,...,4.080000e-06,3.070000e-06,70,1,1,1,1,1,LC group,4
3,3.780000e-07,4.190000e-07,0.000241,1.100000e-06,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000002,...,1.420000e-06,1.070000e-06,66,0,0,4,1,1,LC group,3
4,4.260000e-07,4.780000e-07,0.000276,1.440000e-06,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000003,...,1.400000e-06,1.190000e-06,69,1,1,1,1,1,LC group,4


,S1_T1,S2_T1,S3_T1,S4_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S4_T2,S5_T2,S6_T2,S1_T3,S2_T3,S3_T3,S4_T3,S5_T3,S6_T3
0,3.180000e-07,3.730000e-07,0.000230,9.420000e-07,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000004,0.000002,1.500000e-06,5.560000e-07,2.720000e-07,0.000035,0.000004,8.850000e-07,7.660000e-07
1,3.300000e-07,2.770000e-07,0.000268,1.120000e-06,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000002,0.000001,8.760000e-07,2.110000e-07,1.750000e-07,0.000035,0.000004,1.160000e-06,9.310000e-07
2,5.450000e-07,4.810000e-07,0.000527,2.400000e-06,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000005,0.000003,2.750000e-06,6.980000e-07,5.250000e-07,0.000119,0.000013,4.080000e-06,3.070000e-06
3,3.780000e-07,4.190000e-07,0.000241,1.100000e-06,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000002,0.000001,9.480000e-07,2.760000e-07,3.300000e-07,0.000038,0.000005,1.420000e-06,1.070000e-06
4,4.260000e-07,4.780000e-07,0.000276,1.440000e-06,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000003,0.000001,1.170000e-06,3.210000e-07,3.700000e-07,0.000044,0.000006,1.400000e-06,1.190000e-06


,Group
0,1
1,1
2,1
3,1
4,1


LOGISTIC REGRESSION

For sensor only

In [19]:
#With SMOTE


nested_cv_corrected(model_name='LogisticRegression', X=X, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)




LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (6.4s)  params={'model__C': np.float64(7.158728631500198), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.9441  Acc=0.9583  Sens=0.9231  Spec=1.0000  (1.3s)  params={'model__C': np.float64(0.4108318894699929), 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.9580  Acc=0.9583  Sens=1.0000  Spec=0.9091  (1.3s)  params={'model__C': np.float64(3.4220529032706923), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.9000  Acc=0.9130  Sens=0.8462  Spec=1.0000  (1.4s)  params={'model__C': np.float64(27.293781650374736), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.9769  Acc=0.9565  Sens=0.9231  Spec=1.0000  (2.4s)  params={'model__C': np.float64(1.01291979568

{'roc_auc': [np.float64(0.993006993006993),
  np.float64(0.9440559440559441),
  np.float64(0.958041958041958),
  np.float64(0.9),
  np.float64(0.9769230769230769)],
 'accuracy': [0.9583333333333334,
  0.9583333333333334,
  0.9583333333333334,
  0.9130434782608695,
  0.9565217391304348],
 'sensitivity': [np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(1.0),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(1.0)],
 'best_params': [{'model__C': np.float64(7.158728631500198),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(0.4108318894699929),
   'model__class_weight': None,
   'model__penalty': 'l2',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(3.4220529032706923),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'model__solv

In [21]:
#Without SMOTE
nested_cv_corrected(model_name='LogisticRegression', X=X, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (1.4s)  params={'model__C': np.float64(7.158728631500198), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.9650  Acc=0.9583  Sens=0.9231  Spec=1.0000  (2.0s)  params={'model__C': np.float64(0.5414413211338525), 'model__class_weight': 'balanced', 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.9650  Acc=0.9167  Sens=0.9231  Spec=0.9091  (1.5s)  params={'model__C': np.float64(3.4220529032706923), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.9000  Acc=0.9130  Sens=0.8462  Spec=1.0000  (1.1s)  params={'model__C': np.float64(27.293781650374736), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.9692  Acc=0.9565  Sens=0.9231  Spec=1.0000  (1.1s)  params={'model__C': np.float64(1.09074

{'roc_auc': [np.float64(0.993006993006993),
  np.float64(0.965034965034965),
  np.float64(0.965034965034965),
  np.float64(0.9),
  np.float64(0.9692307692307692)],
 'accuracy': [0.9583333333333334,
  0.9583333333333334,
  0.9166666666666666,
  0.9130434782608695,
  0.9565217391304348],
 'sensitivity': [np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(1.0)],
 'best_params': [{'model__C': np.float64(7.158728631500198),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(0.5414413211338525),
   'model__class_weight': 'balanced',
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(3.4220529032706923),
   'model__class_weight': None,
   'model__penalty': '

For demographics only

In [22]:
X1 = df[["Age","Sex","Smoking"]]


In [23]:
#with SMOTE
nested_cv_corrected(model_name='LogisticRegression', X=X1, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)


LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.8392  Acc=0.7917  Sens=0.8462  Spec=0.7273  (1.1s)  params={'model__C': np.float64(67.32248920775338), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.8636  Acc=0.7083  Sens=0.6923  Spec=0.7273  (1.1s)  params={'model__C': np.float64(14.528246637516036), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.7063  Acc=0.6250  Sens=0.8462  Spec=0.3636  (1.1s)  params={'model__C': np.float64(0.061991000078022655), 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.6808  Acc=0.5652  Sens=0.3846  Spec=0.8000  (1.1s)  params={'model__C': np.float64(0.9163741808778786), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.8577  Acc=0.7826  Sens=0.6154  Spec=1.0000  (1.1s)  params={'model__C': np.float64

{'roc_auc': [np.float64(0.8391608391608392),
  np.float64(0.8636363636363635),
  np.float64(0.7062937062937062),
  np.float64(0.6807692307692308),
  np.float64(0.8576923076923078)],
 'accuracy': [0.7916666666666666,
  0.7083333333333334,
  0.625,
  0.5652173913043478,
  0.782608695652174],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(0.6923076923076923),
  np.float64(0.8461538461538461),
  np.float64(0.38461538461538464),
  np.float64(0.6153846153846154)],
 'specificity': [np.float64(0.7272727272727273),
  np.float64(0.7272727272727273),
  np.float64(0.36363636363636365),
  np.float64(0.8),
  np.float64(1.0)],
 'best_params': [{'model__C': np.float64(67.32248920775338),
   'model__class_weight': 'balanced',
   'model__penalty': 'l2',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(14.528246637516036),
   'model__class_weight': 'balanced',
   'model__penalty': 'l2',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(0.061991000078022655),
   'model_

In [24]:
#without SMOTE
nested_cv_corrected(model_name='LogisticRegression', X=X1, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.8392  Acc=0.7917  Sens=0.8462  Spec=0.7273  (0.9s)  params={'model__C': np.float64(14.528246637516036), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.8427  Acc=0.7083  Sens=0.6923  Spec=0.7273  (0.9s)  params={'model__C': np.float64(67.32248920775338), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.7133  Acc=0.6667  Sens=0.9231  Spec=0.3636  (1.4s)  params={'model__C': np.float64(0.19069966103000435), 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.6923  Acc=0.5217  Sens=0.3846  Spec=0.7000  (1.5s)  params={'model__C': np.float64(4.5705630998014515), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.8577  Acc=0.7826  Sens=0.6923  Spec=0.9000  (1.3s)  params={'model__C': np.float64(

{'roc_auc': [np.float64(0.8391608391608392),
  np.float64(0.8426573426573426),
  np.float64(0.7132867132867133),
  np.float64(0.6923076923076923),
  np.float64(0.8576923076923078)],
 'accuracy': [0.7916666666666666,
  0.7083333333333334,
  0.6666666666666666,
  0.5217391304347826,
  0.782608695652174],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(0.6923076923076923),
  np.float64(0.9230769230769231),
  np.float64(0.38461538461538464),
  np.float64(0.6923076923076923)],
 'specificity': [np.float64(0.7272727272727273),
  np.float64(0.7272727272727273),
  np.float64(0.36363636363636365),
  np.float64(0.7),
  np.float64(0.9)],
 'best_params': [{'model__C': np.float64(14.528246637516036),
   'model__class_weight': 'balanced',
   'model__penalty': 'l2',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(67.32248920775338),
   'model__class_weight': 'balanced',
   'model__penalty': 'l2',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(0.19069966103000435)

For combined model

In [25]:
X2 = df.drop(columns=["Group","LC_stage","LC_type","lscm","Group_name"])

In [26]:
#with SMOTE
nested_cv_corrected(model_name='LogisticRegression', X=X2, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)


LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (1.3s)  params={'model__C': np.float64(4.5705630998014515), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.9720  Acc=0.9167  Sens=0.9231  Spec=0.9091  (1.4s)  params={'model__C': np.float64(0.5414413211338525), 'model__class_weight': 'balanced', 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.9441  Acc=0.8750  Sens=0.9231  Spec=0.8182  (1.4s)  params={'model__C': np.float64(3.4220529032706923), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.8846  Acc=0.9130  Sens=0.8462  Spec=1.0000  (1.3s)  params={'model__C': np.float64(67.32248920775338), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.9692  Acc=0.9565  Sens=0.9231  Spec=1.0000  (1.3s)  params={'model__C': np.float64(1

{'roc_auc': [np.float64(0.993006993006993),
  np.float64(0.9720279720279721),
  np.float64(0.944055944055944),
  np.float64(0.8846153846153847),
  np.float64(0.9692307692307692)],
 'accuracy': [0.9583333333333334,
  0.9166666666666666,
  0.875,
  0.9130434782608695,
  0.9565217391304348],
 'sensitivity': [np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(0.8181818181818182),
  np.float64(1.0),
  np.float64(1.0)],
 'best_params': [{'model__C': np.float64(4.5705630998014515),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(0.5414413211338525),
   'model__class_weight': 'balanced',
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(3.4220529032706923),
   'model__class_weight': None,
   

In [27]:
#without SMOTE
nested_cv_corrected(model_name='LogisticRegression', X=X2, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


LogisticRegression: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=1.0000  Acc=0.9167  Sens=0.8462  Spec=1.0000  (1.1s)  params={'model__C': np.float64(1.0129197956845732), 'model__class_weight': 'balanced', 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 2: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (1.1s)  params={'model__C': np.float64(3.4220529032706923), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 3: ROC-AUC=0.9441  Acc=0.8750  Sens=0.9231  Spec=0.8182  (1.2s)  params={'model__C': np.float64(3.4220529032706923), 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
  Fold 4: ROC-AUC=0.8846  Acc=0.9130  Sens=0.8462  Spec=1.0000  (2.0s)  params={'model__C': np.float64(14.528246637516036), 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'liblinear'}
  Fold 5: ROC-AUC=0.9385  Acc=0.9565  Sens=0.9231  Spec=1.0000  (1.7s)  params={'model__C': np.float64(

{'roc_auc': [np.float64(1.0),
  np.float64(0.993006993006993),
  np.float64(0.944055944055944),
  np.float64(0.8846153846153847),
  np.float64(0.9384615384615385)],
 'accuracy': [0.9166666666666666,
  0.9583333333333334,
  0.875,
  0.9130434782608695,
  0.9565217391304348],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(1.0),
  np.float64(0.8181818181818182),
  np.float64(1.0),
  np.float64(1.0)],
 'best_params': [{'model__C': np.float64(1.0129197956845732),
   'model__class_weight': 'balanced',
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(3.4220529032706923),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'model__solver': 'liblinear'},
  {'model__C': np.float64(3.4220529032706923),
   'model__class_weight': None,
   'model__penalty': 'l1',
   'mo

RANDOM FOREST

For sensor only

In [28]:
#with SMOITE
nested_cv_corrected(model_name='RandomForest', X=X, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=1.0000  Acc=1.0000  Sens=1.0000  Spec=1.0000  (78.3s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 2: ROC-AUC=0.9510  Acc=0.8333  Sens=0.7692  Spec=0.9091  (78.9s)  params={'model__class_weight': None, 'model__max_depth': 4, 'model__max_features': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 3, 'model__n_estimators': 269}
  Fold 3: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (77.1s)  params={'model__class_weight': 'balanced', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 445}
  Fold 4: ROC-AUC=0.9000  Acc=0.8696  Sens=0.8462  Spec=0.9000  (75.7s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_s

{'roc_auc': [np.float64(1.0),
  np.float64(0.951048951048951),
  np.float64(0.993006993006993),
  np.float64(0.8999999999999999),
  np.float64(0.9615384615384616)],
 'accuracy': [1.0,
  0.8333333333333334,
  0.9583333333333334,
  0.8695652173913043,
  0.9565217391304348],
 'sensitivity': [np.float64(1.0),
  np.float64(0.7692307692307693),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(0.9),
  np.float64(1.0)],
 'best_params': [{'model__class_weight': None,
   'model__max_depth': None,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 2,
   'model__n_estimators': 382},
  {'model__class_weight': None,
   'model__max_depth': 4,
   'model__max_features': None,
   'model__min_samples_leaf': 4,
   'model__min_samples_split': 3,
   'model__n_estimators': 269},
  {'model__class_weight': 'balan

In [29]:
#without SMOTE
nested_cv_corrected(model_name='RandomForest', X=X, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=1.0000  Acc=1.0000  Sens=1.0000  Spec=1.0000  (76.5s)  params={'model__class_weight': None, 'model__max_depth': 5, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 4, 'model__n_estimators': 246}
  Fold 2: ROC-AUC=0.9930  Acc=0.9167  Sens=0.9231  Spec=0.9091  (74.6s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 3: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (75.8s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 4: ROC-AUC=0.9000  Acc=0.8696  Sens=0.8462  Spec=0.9000  (73.9s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples

{'roc_auc': [np.float64(1.0),
  np.float64(0.993006993006993),
  np.float64(0.993006993006993),
  np.float64(0.8999999999999999),
  np.float64(0.9615384615384616)],
 'accuracy': [1.0,
  0.9166666666666666,
  0.9583333333333334,
  0.8695652173913043,
  0.9130434782608695],
 'sensitivity': [np.float64(1.0),
  np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.8461538461538461)],
 'specificity': [np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(0.9),
  np.float64(1.0)],
 'best_params': [{'model__class_weight': None,
   'model__max_depth': 5,
   'model__max_features': None,
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 4,
   'model__n_estimators': 246},
  {'model__class_weight': None,
   'model__max_depth': None,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 2,
   'model__n_estimators': 382},
  {'model__class_weight': None,


For demographics only

In [30]:
#with SMOTE
nested_cv_corrected(model_name='RandomForest', X=X1, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.8322  Acc=0.8333  Sens=0.8462  Spec=0.8182  (71.8s)  params={'model__class_weight': None, 'model__max_depth': 2, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 9, 'model__n_estimators': 161}
  Fold 2: ROC-AUC=0.9126  Acc=0.7500  Sens=0.9231  Spec=0.5455  (72.9s)  params={'model__class_weight': None, 'model__max_depth': 2, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 9, 'model__n_estimators': 161}
  Fold 3: ROC-AUC=0.7203  Acc=0.6250  Sens=0.8462  Spec=0.3636  (72.1s)  params={'model__class_weight': 'balanced', 'model__max_depth': 2, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 458}
  Fold 4: ROC-AUC=0.6769  Acc=0.5652  Sens=0.6154  Spec=0.5000  (73.8s)  params={'model__class_weight': None, 'model__max_depth': 2, 'model__max_features': None, 'model__min_samples_le

{'roc_auc': [np.float64(0.8321678321678322),
  np.float64(0.9125874125874125),
  np.float64(0.7202797202797203),
  np.float64(0.6769230769230768),
  np.float64(0.7653846153846154)],
 'accuracy': [0.8333333333333334,
  0.75,
  0.625,
  0.5652173913043478,
  0.6956521739130435],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.6153846153846154),
  np.float64(0.5384615384615384)],
 'specificity': [np.float64(0.8181818181818182),
  np.float64(0.5454545454545454),
  np.float64(0.36363636363636365),
  np.float64(0.5),
  np.float64(0.9)],
 'best_params': [{'model__class_weight': None,
   'model__max_depth': 2,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 9,
   'model__n_estimators': 161},
  {'model__class_weight': None,
   'model__max_depth': 2,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 9,
   'model__n_e

In [31]:
#without SMOTE
nested_cv_corrected(model_name='RandomForest', X=X1, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=0.8462  Acc=0.7917  Sens=0.8462  Spec=0.7273  (72.0s)  params={'model__class_weight': 'balanced', 'model__max_depth': 2, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 458}
  Fold 2: ROC-AUC=0.9126  Acc=0.7500  Sens=0.8462  Spec=0.6364  (73.6s)  params={'model__class_weight': 'balanced', 'model__max_depth': 2, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 458}
  Fold 3: ROC-AUC=0.7273  Acc=0.6250  Sens=0.9231  Spec=0.2727  (71.7s)  params={'model__class_weight': None, 'model__max_depth': 2, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 9, 'model__n_estimators': 161}
  Fold 4: ROC-AUC=0.6577  Acc=0.5652  Sens=0.6154  Spec=0.5000  (72.0s)  params={'model__class_weight': None, 'model__max_depth': 2, 'model__max_features': None, 'model__min_samp

{'roc_auc': [np.float64(0.8461538461538461),
  np.float64(0.9125874125874126),
  np.float64(0.7272727272727273),
  np.float64(0.6576923076923077),
  np.float64(0.7884615384615384)],
 'accuracy': [0.7916666666666666,
  0.75,
  0.625,
  0.5652173913043478,
  0.6956521739130435],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231),
  np.float64(0.6153846153846154),
  np.float64(0.6153846153846154)],
 'specificity': [np.float64(0.7272727272727273),
  np.float64(0.6363636363636364),
  np.float64(0.2727272727272727),
  np.float64(0.5),
  np.float64(0.8)],
 'best_params': [{'model__class_weight': 'balanced',
   'model__max_depth': 2,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 4,
   'model__n_estimators': 458},
  {'model__class_weight': 'balanced',
   'model__max_depth': 2,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 4,
   

For combined model

In [32]:
#with SMOTE
nested_cv_corrected(model_name='RandomForest', X=X2, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= True)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=1.0000  Acc=1.0000  Sens=1.0000  Spec=1.0000  (75.3s)  params={'model__class_weight': 'balanced', 'model__max_depth': 5, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 409}
  Fold 2: ROC-AUC=0.9441  Acc=0.7917  Sens=0.6923  Spec=0.9091  (76.4s)  params={'model__class_weight': 'balanced', 'model__max_depth': 6, 'model__max_features': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 9, 'model__n_estimators': 313}
  Fold 3: ROC-AUC=0.9930  Acc=0.9583  Sens=0.9231  Spec=1.0000  (76.4s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 4: ROC-AUC=0.8846  Acc=0.8696  Sens=0.8462  Spec=0.9000  (74.7s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__mi

{'roc_auc': [np.float64(1.0),
  np.float64(0.9440559440559441),
  np.float64(0.993006993006993),
  np.float64(0.8846153846153847),
  np.float64(0.9692307692307692)],
 'accuracy': [1.0,
  0.7916666666666666,
  0.9583333333333334,
  0.8695652173913043,
  0.9565217391304348],
 'sensitivity': [np.float64(1.0),
  np.float64(0.6923076923076923),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(0.9),
  np.float64(1.0)],
 'best_params': [{'model__class_weight': 'balanced',
   'model__max_depth': 5,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 5,
   'model__n_estimators': 409},
  {'model__class_weight': 'balanced',
   'model__max_depth': 6,
   'model__max_features': None,
   'model__min_samples_leaf': 4,
   'model__min_samples_split': 9,
   'model__n_estimators': 313},
  {'model__class_weigh

In [33]:
#without SMOTE
nested_cv_corrected(model_name='RandomForest', X=X2, y=y, n_inner_splits=3, n_outer_splits=5, random_state=42, use_smote= False)


RandomForest: Nested CV (5x3, corrected pipeline)
  Fold 1: ROC-AUC=1.0000  Acc=1.0000  Sens=1.0000  Spec=1.0000  (75.4s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 2: ROC-AUC=0.9930  Acc=0.9167  Sens=0.9231  Spec=0.9091  (74.5s)  params={'model__class_weight': 'balanced', 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 445}
  Fold 3: ROC-AUC=1.0000  Acc=0.9583  Sens=0.9231  Spec=1.0000  (75.6s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 382}
  Fold 4: ROC-AUC=0.8885  Acc=0.8696  Sens=0.8462  Spec=0.9000  (74.0s)  params={'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__

{'roc_auc': [np.float64(1.0),
  np.float64(0.993006993006993),
  np.float64(1.0),
  np.float64(0.8884615384615385),
  np.float64(0.9615384615384616)],
 'accuracy': [1.0,
  0.9166666666666666,
  0.9583333333333334,
  0.8695652173913043,
  0.9565217391304348],
 'sensitivity': [np.float64(1.0),
  np.float64(0.9230769230769231),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.9230769230769231)],
 'specificity': [np.float64(1.0),
  np.float64(0.9090909090909091),
  np.float64(1.0),
  np.float64(0.9),
  np.float64(1.0)],
 'best_params': [{'model__class_weight': None,
   'model__max_depth': None,
   'model__max_features': 'sqrt',
   'model__min_samples_leaf': 1,
   'model__min_samples_split': 2,
   'model__n_estimators': 382},
  {'model__class_weight': 'balanced',
   'model__max_depth': None,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 2,
   'model__n_estimators': 445},
  {'model__class_weight': None,
   